# Research 70 — RealMLP-TD OOF on CV split seed 42 (Vast.ai GPU)

Reproducibility check. RealMLP-TD on CV split seed 42 on the remote GPU, to compare with the local run in notebook 65. The output file has a separate name, so notebook 65's file is not overwritten.

Setup is the same as notebook 65: RealMLP-TD defaults (`random_state=42`), the tuned feature set, and median imputation of numeric columns fitted on each fold's training rows only. The only change is the CV split seed. Runs on the remote RTX 3090 instance (torch 2.14.1 cu130, pytabkit 1.8.0, pandas 3.1.0, numpy 2.5.4, scikit-learn 1.9.1). Train OOF only; `test.csv` is not loaded. The comparison with the submitted recipe is notebook 71, which runs locally after these outputs are copied back.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pytabkit import RealMLP_TD_Classifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(train, y))
print("split seed", SEED, "| rows", len(train), "| fold sizes", [len(v) for _, v in folds])

/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


split seed 42 | rows 104000 | fold sizes [34667, 34667, 34666]


In [2]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (a, b) in ratios.items():
        x[name] = x[a] / (x[b].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("features", X.shape[1], "categorical", len(cat_cols), "numeric", len(num_cols))

features 51 categorical 10 numeric 41


In [3]:
p_rm = np.zeros((len(y), 4), dtype=float)
fold_scores = []
started = time.perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    x_fit = X.iloc[fit_idx].copy()
    x_valid = X.iloc[valid_idx].copy()
    x_fit[cat_cols] = x_fit[cat_cols].fillna("__MISSING__").astype(str)
    x_valid[cat_cols] = x_valid[cat_cols].fillna("__MISSING__").astype(str)
    medians = x_fit[num_cols].median()
    x_fit[num_cols] = x_fit[num_cols].fillna(medians)
    x_valid[num_cols] = x_valid[num_cols].fillna(medians)
    model = RealMLP_TD_Classifier(device="cuda", random_state=42, n_threads=4, verbosity=0)
    model.fit(x_fit, y[fit_idx], cat_col_names=cat_cols)
    p_rm[valid_idx] = model.predict_proba(x_valid)
    score = f1_score(y[valid_idx], p_rm[valid_idx].argmax(1), average="macro")
    fold_scores.append(score)
    print(f"Fold {fold}: RealMLP-TD macro F1={score:.6f}", flush=True)
print(f"Elapsed seconds={time.perf_counter() - started:.1f}. Train OOF only; Test not loaded.")
pd.DataFrame(p_rm, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions" / "segment-realmlp-td-seed42-remote-oof.csv", index=False
)

/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 1: RealMLP-TD macro F1=0.752102


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 2: RealMLP-TD macro F1=0.754464


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fold 3: RealMLP-TD macro F1=0.757851


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Elapsed seconds=1439.2. Train OOF only; Test not loaded.


In [4]:
summary = pd.DataFrame(
    [
        {
            "split_seed": SEED,
            "macro_f1": f1_score(y, p_rm.argmax(1), average="macro"),
            "weighted_f1": f1_score(y, p_rm.argmax(1), average="weighted"),
            **{f"fold_{k}_macro_f1": s for k, s in enumerate(fold_scores, start=1)},
        }
    ]
)
display(summary.round(6))
summary.to_csv(ROOT / "assets" / "segment-realmlp-td-seed42-remote-results.csv", index=False)

,split_seed,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1
0,42,0.754808,0.76821,0.752102,0.754464,0.757851


## Research decision

No decision here. The predeclared rule is applied in notebook 71 after all split runs are copied back.